# Task 2: Tự thu thập Sentinel-2 + RPG theo slide 2

**Mục tiêu: ~25.000 mẫu địa lý khác nhau, mùa 2020-2021 (ảnh 01/10/2020 → 30/09/2021, nhãn RPG 2021).** Report ghi số thực tế và `submission_target_met`.

Bản tối ưu cho Kaggle: nhiều khối được tải song song (`BLOCK_WORKERS`) trong khi luồng chính ghi pickle/SQLite; các vùng được xen kẽ để mẫu trải đều địa lý; cache tạm không nén và chỉ theo ngày; báo cáo tổng hợp ghi định kỳ; log gọn; dừng an toàn theo `TIME_BUDGET_HOURS` để còn thời gian kiểm tra và đóng gói.

Luồng: vùng/mùa → Sentinel-2 L2A COG → 10 băng + mask → lưới 10 m → chuỗi thời gian/cửa sổ 24×24; RPG cùng vùng/năm → rasterize/cắt cùng cửa sổ → **Normalize → DOY/365.0001 → concat 11 kênh → cut/pad 60 → THWC → cặp input–target**.

Số ngày được chọn **ngẫu nhiên từ 8 đến 12 cho từng khối**, rồi chọn các ngày chụp thật không lặp trong mùa. Các cửa sổ của cùng khối chia sẻ lịch ảnh, như các cửa sổ cắt từ cùng patch PASTIS. Seed và ngày cụ thể được lưu; chạy lại cùng config/cache cho cùng lựa chọn. Không bốc ngày không có ảnh hoặc đổi năm nhãn tùy ý.

Mọi mẫu đều đi qua đầy đủ tiền xử lý khi kiểm tra. Raw pickle giữ ảnh/ngày/nhãn; `prepared_sample_example.npz` chứa cặp input–target cuối cùng. Bật `SAVE_ALL_PREPARED` nếu cần lưu cặp đã chuẩn bị cho tất cả mẫu. Padding không tạo thêm ảnh thật hay mẫu.

Bật Internet; không cần GPU/API key. Ảnh dùng nguồn COG thay THEIA/GEODES và polygon RPG dùng WFS/GeoJSON thay Shapefile; các bước của hai nhánh được giữ.

Nguồn: [Sentinel-2 COG](https://registry.opendata.aws/sentinel-2-l2a-cogs/), [RPG IGN](https://apicarto.ign.fr/api/doc/rpg), [PASTIS24](https://github.com/michaeltrs/DeepSatModels/blob/main/README_TSVIT.md).


## 1. Thư viện
Chỉ cài những thư viện còn thiếu. Không cần API key hoặc torch.

In [ ]:
import importlib.util, subprocess, sys
needed = {"httpx": "httpx>=0.27,<1", "numpy": "numpy>=1.26,<3",
          "rasterio": "rasterio>=1.4,<2", "pyproj": "pyproj>=3.6,<4", "PIL": "Pillow>=10,<13"}
missing = [spec for name,spec in needed.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
print("Dependencies ready")

## 2. Code thu thập và tiền xử lý độc lập
Code nhúng từ `collector.py`, không cần clone repo. `prepare_sample()` tạo đúng cặp input–nhãn theo slide. Nhãn ngoài RPG/mã chưa hỗ trợ là void=19; chỉ tạo ma trận semantic mà TSViT dùng qua `labels[0]`.


In [ ]:
"""Collect original Sentinel-2 COG + IGN/RPG pairs, independently of the web app.

One sample is a distinct 24x24 geographic window containing several real dates.
Raw image values, dates, validity and actual RPG labels are saved; normalization
and temporal padding are performed when preparing model input, not on disk.
"""
from __future__ import annotations

import argparse
from collections import Counter
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, as_completed, wait
import csv
from datetime import date, datetime, timezone
import gzip
import hashlib
from itertools import zip_longest
import json
import math
from pathlib import Path
import pickle
import shutil
import sqlite3
import time
from urllib.parse import urlsplit
import zipfile

import httpx
import numpy as np
from PIL import Image, ImageDraw
from pyproj import Transformer
import rasterio
from rasterio.enums import Resampling
from rasterio.features import geometry_mask, rasterize
from rasterio.transform import array_bounds, from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds, transform_geom


STAC = 'https://earth-search.aws.element84.com/v1'
WFS = 'https://data.geopf.fr/wfs/ows'
BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
ASSETS = ['blue', 'green', 'red', 'rededge1', 'rededge2', 'rededge3',
          'nir', 'nir08', 'swir16', 'swir22']
CLASS_NAMES = ['Background', 'Meadow', 'Soft winter wheat', 'Corn', 'Winter barley',
               'Winter rapeseed', 'Spring barley', 'Sunflower', 'Grapevine', 'Beet',
               'Winter triticale', 'Winter durum wheat', 'Fruits, vegetables, flowers',
               'Potatoes', 'Leguminous fodder', 'Soybeans', 'Orchard', 'Mixed cereal', 'Sorghum']
# Conservative, explicit grouping. This is not the original PASTIS annotation
# recipe. Unsupported crop codes and land outside RPG are void (19), not background.
CROP_MAP = {'PPH': 1, 'PTR': 1, 'PRL': 1, 'BTH': 2, 'MIS': 3, 'MIE': 3,
            'MID': 3, 'ORH': 4, 'CZH': 5, 'ORP': 6, 'TRN': 7,
            'TTH': 10, 'BDH': 11, 'SOJ': 15, 'MCR': 17, 'SOG': 18}
VOID = 19
MEAN = np.array([1165.9398193359375, 1375.6534423828125, 1429.2191162109375,
                 1764.798828125, 2719.273193359375, 3063.61181640625,
                 3205.90185546875, 3319.109619140625, 2422.904296875,
                 1639.370361328125], dtype=np.float32)[None, :, None, None]
STD = np.array([1942.6156005859375, 1881.9234619140625, 1959.3798828125,
                1867.2239990234375, 1754.5850830078125, 1769.4046630859375,
                1784.860595703125, 1767.7100830078125, 1458.963623046875,
                1299.2833251953125], dtype=np.float32)[None, :, None, None]
ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR',
           CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif', GDAL_HTTP_TIMEOUT='45',
           GDAL_HTTP_CONNECTTIMEOUT='15',
           GDAL_HTTP_MAX_RETRY='3', GDAL_HTTP_RETRY_DELAY='2',
           # Fewer round trips per COG: HTTP/2 multiplexing and merged ranges.
           GDAL_HTTP_VERSION='2', GDAL_HTTP_MULTIPLEX='YES',
           GDAL_HTTP_MERGE_CONSECUTIVE_RANGES='YES', AWS_NO_SIGN_REQUEST='YES',
           GDAL_CACHEMAX=512, VSI_CACHE=True, VSI_CACHE_SIZE=16777216)
SOURCES = {
    'imagery': 'https://registry.opendata.aws/sentinel-2-l2a-cogs/',
    'catalog': STAC,
    'labels': WFS,
    'rpg_documentation': 'https://apicarto.ign.fr/api/doc/rpg',
    'crop_codes': 'https://geoservices.ign.fr/sites/default/files/2021-11/DC_DL_RPG_2-0.pdf',
    'reference_format': 'https://github.com/michaeltrs/DeepSatModels/blob/main/README_TSVIT.md',
}


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    temporary.replace(path)


def emit(stage, **values):
    print(json.dumps({'stage': stage, **values}, ensure_ascii=False), flush=True)


def season_interval(season):
    parts = season.replace('–', '-').split('-')
    if len(parts) != 2 or not all(p.isdigit() for p in parts):
        raise ValueError('Season must be YYYY-YYYY.')
    start, end = map(int, parts)
    if end != start + 1:
        raise ValueError('A season must span two consecutive years.')
    return f'{start}-10-01T00:00:00Z/{end}-09-30T23:59:59Z'


def observation_limits(value):
    """A fixed count or an inclusive [minimum, maximum] of real dates."""
    values = [value, value] if isinstance(value, int) and not isinstance(value, bool) else value
    if (not isinstance(values, list) or len(values) != 2
            or any(type(v) is not int for v in values) or not 1 <= values[0] <= values[1] <= 366):
        raise ValueError('observations must be 1..366 or [minimum, maximum] in that range.')
    return tuple(values)


def seeded_rng(seed, *identity):
    digest = hashlib.sha256(json.dumps([seed, *identity], sort_keys=True).encode()).digest()
    return np.random.default_rng(int.from_bytes(digest[:16], 'big'))


def validate_config(config):
    season_interval(config['season'])
    if config['pixel_size_m'] != 10 or config['window_size'] != 24:
        raise ValueError('This PASTIS24 export requires 10 m pixels and 24x24 windows.')
    if int(config['label_year']) != int(config['season'].replace('–', '-').split('-')[1]):
        raise ValueError('Label year must match the crop year ending this season.')
    observation_limits(config['observations'])
    if config.get('date_sampling', 'low_cloud_stratified') not in ('random', 'low_cloud_stratified'):
        raise ValueError('date_sampling must be random or low_cloud_stratified.')
    if type(config.get('seed', 42)) is not int or config.get('seed', 42) < 0:
        raise ValueError('seed must be a nonnegative integer.')
    for key in ['target_samples', 'block_windows', 'workers', 'min_label_pixels']:
        if not isinstance(config[key], int) or config[key] < 1:
            raise ValueError(f'{key} must be a positive integer.')
    if config['block_windows'] > 64 or config['workers'] > 16:
        raise ValueError('Limits: 64 windows per block side, 16 workers.')
    if not 1 <= config['min_label_pixels'] <= 576:
        raise ValueError('min_label_pixels must be between 1 and 576.')
    ids = [r['id'] for r in config['regions']]
    if len(ids) != len(set(ids)) or not ids:
        raise ValueError('Region IDs must be unique and nonempty.')
    for r in config['regions']:
        if not r['id'].replace('_', '').isalnum():
            raise ValueError('Use letters, numbers or underscores in region IDs.')
        lon, lat = r['center']
        if not (-180 <= lon <= 180 and -80 < lat < 84 and 0 < r['size_m'] <= 100000):
            raise ValueError('Invalid region center or size.')


def region_grid(region):
    lon, lat = region['center']
    zone = min(60, max(1, int((lon + 180) // 6) + 1))
    epsg = (32600 if lat >= 0 else 32700) + zone
    x, y = Transformer.from_crs(4326, epsg, always_xy=True).transform(lon, lat)
    side = math.ceil(region['size_m'] / 240) * 24
    west = math.floor((x - side * 5) / 240) * 240
    north = math.ceil((y + side * 5) / 240) * 240
    return {'crs': f'EPSG:{epsg}', 'west': west, 'north': north,
            'height': side, 'width': side, 'region': region['id']}


def grid_transform(g):
    return from_origin(g['west'], g['north'], 10, 10)


def grid_geometry(g):
    west, south, east, north = array_bounds(g['height'], g['width'], grid_transform(g))
    geometry = {'type': 'Polygon', 'coordinates': [[[west, north], [east, north],
                [east, south], [west, south], [west, north]]]}
    return transform_geom(g['crs'], 'EPSG:4326', geometry)


def blocks_for_region(region, block_windows):
    g = region_grid(region)
    step = block_windows * 24
    origins = [(r, c) for r in range(0, g['height'], step) for c in range(0, g['width'], step)]
    origins.sort(key=lambda rc: ((rc[0] + step / 2 - g['height'] / 2) ** 2 +
                                 (rc[1] + step / 2 - g['width'] / 2) ** 2, rc))
    for row, col in origins:
        block = dict(g, west=g['west'] + col * 10, north=g['north'] - row * 10,
                     height=min(step, g['height'] - row), width=min(step, g['width'] - col))
        block['id'] = f"{region['id']}_{row}_{col}"
        yield block


def interleaved_blocks(config):
    """Alternate regions so a partial or time-limited run stays geographically spread."""
    generators = [blocks_for_region(r, config['block_windows']) for r in config['regions']]
    for group in zip_longest(*generators):
        yield from (g for g in group if g is not None)


def collection_plan(config):
    validate_config(config)
    minimum, maximum = observation_limits(config['observations'])
    regions = []
    for r in config['regions']:
        g = region_grid(r)
        regions.append({'id': r['id'], 'candidate_windows': g['height'] * g['width'] // 576,
                        'area_km2': g['height'] * g['width'] / 10000,
                        'blocks': sum(1 for _ in blocks_for_region(r, config['block_windows']))})
    return {'target_samples': config['target_samples'], 'observations': config['observations'],
            'date_sampling': config.get('date_sampling', 'low_cloud_stratified'),
            'seed': config.get('seed', 42),
            'candidate_windows_before_screening': sum(r['candidate_windows'] for r in regions),
            'raw_image_gb_at_target': config['target_samples'] * maximum * 10 * 576 * 4 / 1e9,
            'raw_image_gb_range_at_target': [config['target_samples'] * t * 10 * 576 * 4 / 1e9
                                              for t in (minimum, maximum)],
            'prepared_storage': 'Prepared pairs are streamed by default; exporting all adds disk usage.',
            'network_bytes': 'Not measured or guaranteed by this plan; COG block overhead applies.',
            'regions': regions}


def request_json(client, method, url, **kwargs):
    for attempt in range(4):
        try:
            response = client.request(method, url, **kwargs)
            if response.status_code in (429, 500, 502, 503, 504) and attempt < 3:
                header = response.headers.get('retry-after', '')
                delay = min(60, max(1, int(header))) if header.isdigit() else 2 ** (attempt + 1)
                time.sleep(delay)
                continue
            response.raise_for_status()
            return response.json()
        except httpx.TransportError:
            if attempt == 3:
                raise
            time.sleep(2 ** (attempt + 1))
    raise RuntimeError('Request retries exhausted.')


def search_region(client, region, config, cache):
    path = cache / 'catalog' / (region['id'] + '.json.gz')
    if path.exists():
        with gzip.open(path, 'rt') as stream:
            return json.load(stream)
    geometry = grid_geometry(region_grid(region))
    points = np.asarray(geometry['coordinates'][0])
    body = {'collections': ['sentinel-2-c1-l2a', 'sentinel-2-l2a'],
            'bbox': [*points.min(axis=0), *points.max(axis=0)],
            'datetime': season_interval(config['season']), 'limit': 100}
    body['bbox'] = [float(v) for v in body['bbox']]
    items = {}
    url, method = STAC + '/search', 'POST'
    for _ in range(100):
        result = request_json(client, method, url, **({'json': body} if method == 'POST' else {}))
        for item in result.get('features', []):
            if all(key in item.get('assets', {}) for key in ASSETS + ['scl']):
                items[(item['collection'], item['id'])] = item
        following = next((x for x in result.get('links', []) if x.get('rel') == 'next'), None)
        if not following:
            break
        url = following['href']
        if urlsplit(url).netloc != urlsplit(STAC).netloc:
            raise ValueError('Unexpected STAC pagination host.')
        method = following.get('method', 'GET').upper()
        if method == 'POST':
            body = {**body, **following.get('body', {})}
    else:
        raise RuntimeError('Catalog pagination limit exceeded.')
    output = sorted(items.values(), key=lambda x: (x['properties']['datetime'], x['id']))
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    with gzip.open(tmp, 'wt') as stream:
        json.dump(output, stream)
    tmp.replace(path)
    emit('catalog', region=region['id'], items=len(output))
    return output


def select_observations(items, g, count, sampling='low_cloud_stratified', seed=42):
    minimum, maximum = observation_limits(count)
    tiles = {}
    for item in items:
        # A tile code is not an acquisition footprint: some historical scenes
        # contain only part of that tile. Check every date before deduplication.
        geom = transform_geom('EPSG:4326', g['crs'], item['geometry'])
        covered = geometry_mask([geom], out_shape=(g['height'], g['width']),
                                transform=grid_transform(g), invert=True)
        if not covered.all():
            continue
        tile = item['properties'].get('grid:code', item.get('collection', 'unknown'))
        tiles.setdefault(tile, []).append(item)
    choices = []
    for tile, entries in tiles.items():
        days = {}
        def rank(item):
            return (item['collection'] == 'sentinel-2-c1-l2a',
                    float(item['properties'].get('s2:processing_baseline', 0)),
                    item['properties'].get('updated', ''))
        for item in sorted(entries, key=lambda x: (x['collection'], x['id'])):
            day = item['properties']['datetime'][:10]
            if day not in days or rank(item) > rank(days[day]):
                days[day] = item
        entries = sorted(days.values(), key=lambda x: x['properties']['datetime'])
        if len(entries) >= minimum:
            rng = seeded_rng(seed, g, str(tile))
            n = int(rng.integers(minimum, min(maximum, len(entries)) + 1))
            if sampling == 'random':
                # Sample real acquisition dates without replacement. Sorting
                # catalog entries and using a grid-specific seed makes resume
                # independent of input order and blocks processed previously.
                indices = sorted(rng.choice(len(entries), size=n, replace=False).tolist())
                selected = [entries[i] for i in indices]
                choices.append((float(rng.random()), str(tile), selected))
                continue
            if sampling != 'low_cloud_stratified':
                raise ValueError('Unknown date sampling method.')
            # Spread actual observations across the season; prefer lower tile
            # cloud cover inside each temporal bin. AOI clouds are recorded only.
            bins = np.array_split(np.arange(len(entries)), n)
            selected = [entries[min(indices, key=lambda i: (
                entries[int(i)]['properties'].get('eo:cloud_cover', 100), int(i)))] for indices in bins]
            choices.append((sum(float(x['properties'].get('eo:cloud_cover', 100)) for x in selected),
                            str(tile), selected))
    if not choices:
        return []
    return min(choices, key=lambda x: (x[0], x[1]))[2]


def fetch_parcels(client, g, year, cache):
    path = cache / 'parcels' / (g['id'] + '.geojson')
    if path.exists():
        return json.loads(path.read_text())
    bounds = array_bounds(g['height'], g['width'], grid_transform(g))
    bbox = transform_bounds(g['crs'], 'EPSG:4326', *bounds, densify_pts=21)
    features, seen, start = [], set(), 0
    for _ in range(100):
        params = {'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature',
                  'typeNames': f'RPG.{year}:parcelles_graphiques',
                  'outputFormat': 'application/json', 'srsName': 'CRS:84',
                  'bbox': ','.join(str(x) for x in bbox) + ',CRS:84',
                  'count': 1000, 'startIndex': start, 'sortBy': 'id_parcel'}
        data = request_json(client, 'GET', WFS, params=params)
        page = data.get('features', [])
        before = len(features)
        for feature in page:
            identity = str(feature['properties'].get('id_parcel', feature.get('id')))
            if identity not in seen:
                seen.add(identity)
                features.append(feature)
        matched = data.get('numberMatched', data.get('totalFeatures'))
        start += len(page)
        if not page or len(page) < 1000 or (str(matched).isdigit() and start >= int(matched)):
            break
        if len(features) == before:
            raise RuntimeError('RPG pagination stopped advancing; incomplete labels rejected.')
        time.sleep(.2)
    else:
        raise RuntimeError('RPG pagination limit exceeded; incomplete labels rejected.')
    output = {'type': 'FeatureCollection', 'features': features,
              'source': WFS, 'layer': f'RPG.{year}:parcelles_graphiques',
              'coordinate_order': 'longitude, latitude (CRS:84)', 'bbox': list(bbox)}
    write_json(path, output)
    return output


def rasterize_parcels(parcels, g):
    semantic_shapes, id_shapes = [], []
    codes = Counter()
    # Numeric local parcel indices are backed by the original ID lookup.
    lookup = {}
    features = sorted(parcels['features'], key=lambda f: str(f['properties'].get('id_parcel', f.get('id'))))
    for index, feature in enumerate(features, 1):
        props = feature['properties']
        code = str(props.get('code_cultu', '')).upper()
        codes[code] += 1
        geometry = transform_geom('EPSG:4326', g['crs'], feature['geometry'])
        semantic_shapes.append((geometry, CROP_MAP.get(code, VOID)))
        id_shapes.append((geometry, index))
        lookup[str(index)] = str(props.get('id_parcel', feature.get('id')))
    shape = (g['height'], g['width'])
    labels = np.full(shape, VOID, dtype=np.uint8)
    ids = np.zeros(shape, dtype=np.int32)
    if semantic_shapes:
        labels = rasterize(semantic_shapes, out_shape=shape, transform=grid_transform(g),
                           fill=VOID, dtype='uint8', all_touched=False)
        ids = rasterize(id_shapes, out_shape=shape, transform=grid_transform(g),
                        fill=0, dtype='int32', all_touched=False)
    return labels, ids, lookup, dict(codes)


def read_asset(item, key, g):
    asset = item['assets'][key]
    url = asset['href']
    parsed = urlsplit(url)
    if (parsed.scheme != 'https' or not parsed.hostname or not parsed.hostname.endswith('.amazonaws.com')
            or not parsed.path.endswith('.tif') or asset.get('storage:requester_pays')):
        raise ValueError('Expected a public HTTPS Sentinel-2 COG asset.')
    info = asset.get('raster:bands', [{}])[0]
    quality = key == 'scl'
    with rasterio.Env(**ENV), rasterio.open(url) as source:
        nodata = info.get('nodata', source.nodata)
        if nodata is None:
            raise ValueError('Band nodata is missing; refusing to invent it.')
        with WarpedVRT(source, crs=g['crs'], transform=grid_transform(g),
                       width=g['width'], height=g['height'], src_nodata=nodata,
                       nodata=nodata, dtype='float32',
                       resampling=Resampling.nearest if quality else Resampling.bilinear) as aligned:
            raw = aligned.read(1)
    valid = np.isfinite(raw) & (raw != nodata)
    if quality:
        return raw.astype(np.uint8), valid
    if 'scale' not in info:
        raise ValueError('Reflectance scale metadata missing; cannot normalize this band safely.')
    image = (raw * np.float32(info['scale']) + np.float32(info.get('offset', 0))) * np.float32(10000)
    image[~valid] = 0
    return image, valid


def observation_key(items, g):
    identity = {'items': [(x['collection'], x['id']) for x in items], 'grid': g, 'format': 2}
    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:24]


def read_observations(items, g, cache, workers, persist_block=True):
    path = cache / 'imagery' / (observation_key(items, g) + '.npz')
    if path.exists():
        with np.load(path, allow_pickle=False) as data:
            return data['image'], data['valid'], data['scl']
    image = np.zeros((len(items), 10, g['height'], g['width']), dtype=np.float32)
    valid = np.ones((len(items), g['height'], g['width']), dtype=bool)
    scl = np.zeros_like(valid, dtype=np.uint8)
    remaining, done_bands = {}, Counter()
    for t, item in enumerate(items):
        per_date = cache / 'imagery_dates' / (observation_key([item], g) + '.npz')
        if per_date.exists():
            with np.load(per_date, allow_pickle=False) as data:
                image[t], valid[t], scl[t] = data['image'], data['valid'], data['scl']
        else:
            remaining[t] = per_date
    with ThreadPoolExecutor(max_workers=workers) as pool:
        tasks = {pool.submit(read_asset, item, key, g): (t, b, key)
                 for t, item in enumerate(items) if t in remaining
                 for b, key in enumerate(ASSETS + ['scl'])}
        # Cancel queued bands if any read fails. Waiting for every queued
        # download after the first failure can otherwise hide a network error
        # for an entire 8--12-date block.
        def cancel_pending_on_error(future):
            if not future.cancelled() and future.exception() is not None:
                t, _, key = tasks[future]
                emit('band_error', item=items[t]['id'], band=key,
                     error=f'{type(future.exception()).__name__}: {future.exception()}')
                for pending in tasks:
                    pending.cancel()
        for future in tasks:
            future.add_done_callback(cancel_pending_on_error)
        for future in as_completed(tasks):
            if future.cancelled():
                continue
            t, band, key = tasks[future]
            values, native = future.result()
            valid[t] &= native
            if key == 'scl':
                scl[t] = values
                valid[t] &= values != 1
            else:
                image[t, band] = values
            done_bands[t] += 1
            if done_bands[t] == len(ASSETS) + 1:
                # Persist complete dates immediately, so an interrupted block
                # can reuse them instead of reading all selected dates again.
                per_date = remaining[t]
                per_date.parent.mkdir(parents=True, exist_ok=True)
                # Uncompressed: this is short-lived scratch, compression costs CPU.
                temporary = per_date.with_suffix('.tmp')
                with temporary.open('wb') as stream:
                    np.savez(stream, image=image[t], valid=valid[t], scl=scl[t])
                temporary.replace(per_date)
    if not persist_block:
        return image, valid, scl
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, image=image, valid=valid, scl=scl)
    tmp.replace(path)
    return image, valid, scl


def noleap_doy(value):
    d = date.fromisoformat(value[:10])
    return date(2001, d.month, min(d.day, 28) if d.month == 2 else d.day).timetuple().tm_yday


def prepare_input(sample):
    """Execute slide 2 explicitly: Normalize -> DOY -> concat -> pad/cut -> THWC."""
    image = np.asarray(sample['img'], dtype=np.float32)
    doy = np.asarray(sample['doy'], dtype=np.float32)
    if image.ndim != 4 or image.shape[1:] != (10, 24, 24) or doy.shape != (len(image),) or not len(image):
        raise ValueError('Expected imagery [T,10,24,24] and T dates.')
    if not np.isfinite(image).all() or not np.isfinite(doy).all() or ((doy < 1) | (doy > 365)).any():
        raise ValueError('Invalid image or acquisition date.')
    normalized = (image - MEAN) / STD
    dates = np.broadcast_to((doy / np.float32(365.0001))[:, None, None, None],
                            (len(image), 1, 24, 24))
    concatenated = np.concatenate((normalized, dates), axis=1)
    padded = np.zeros((60, 11, 24, 24), dtype=np.float32)
    n = min(len(image), 60)
    padded[:n] = concatenated[:n]
    inputs = np.ascontiguousarray(padded.transpose(0, 2, 3, 1))
    trace = {'raw_tchw': list(image.shape), 'normalized_tchw': list(normalized.shape),
             'date_channel_tchw': list(dates.shape), 'concatenated_tchw': list(concatenated.shape),
             'cut_or_pad_tchw': list(padded.shape), 'inputs_thwc': list(inputs.shape),
             'real_observations': len(image), 'retained_observations': n,
             'zero_padding_frames': 60 - n, 'cut_frames': max(0, len(image) - 60),
             'normalization': 'PASTIS fold 1 channel mean/std used by the current checkpoint',
             'date_divisor': 365.0001}
    return inputs, trace


def model_input(sample):
    return prepare_input(sample)[0]


def prepare_sample(sample):
    """Return the paired (U,Y) sample from slide 2, compatible with TSViT transforms."""
    inputs, trace = prepare_input(sample)
    raw_labels = np.asarray(sample['labels'])
    if (raw_labels.ndim != 3 or raw_labels.shape[1:] != (24, 24) or not len(raw_labels)
            or not np.isin(raw_labels[0], np.arange(20)).all()):
        raise ValueError('Expected labels[0] to contain semantic class IDs 0..19 on a 24x24 grid.')
    labels = raw_labels[0].astype(np.float32)[..., None]
    trace['target_hwc'] = list(labels.shape)
    return {'inputs': inputs, 'labels': labels, 'unk_masks': labels != VOID,
            'seq_lengths': min(len(sample['doy']), 60), 'pipeline': trace}


def save_prepared_sample(path, prepared):
    """Compressed file of actual model input U and its paired semantic target Y."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp')
    with temporary.open('wb') as stream:
        np.savez_compressed(stream, inputs=prepared['inputs'], labels=prepared['labels'],
                            unk_masks=prepared['unk_masks'], seq_lengths=np.int16(prepared['seq_lengths']))
    temporary.replace(path)


def open_index(output):
    connection = sqlite3.connect(output / 'index.sqlite3')
    connection.execute('CREATE TABLE IF NOT EXISTS samples (id TEXT PRIMARY KEY, path TEXT NOT NULL, '
                       'bytes INTEGER NOT NULL, metadata TEXT NOT NULL)')
    connection.execute('CREATE TABLE IF NOT EXISTS blocks (id TEXT PRIMARY KEY, status TEXT, message TEXT)')
    connection.commit()
    return connection


def sample_count(connection):
    return connection.execute('SELECT COUNT(*) FROM samples').fetchone()[0]


def export_block(connection, output, g, items, image, valid, scl, labels, parcel_ids,
                 lookup, config, target):
    days = [x['properties']['datetime'][:10] for x in items]
    order = sorted(range(len(days)), key=lambda i: (noleap_doy(days[i]), days[i]))
    dates = [days[i] for i in order]
    image, valid, scl = image[order], valid[order], scl[order]
    emitted = 0
    count = sample_count(connection)
    for row in range(0, g['height'], 24):
        for col in range(0, g['width'], 24):
            if count >= target:
                return emitted, False
            west, north = int(g['west'] + col * 10), int(g['north'] - row * 10)
            sid = f"y{config['label_year']}_{g['crs'].replace(':', '')}_e{west}_n{north}"
            existing = connection.execute('SELECT path, bytes FROM samples WHERE id=?', (sid,)).fetchone()
            if existing:
                existing_path = output / existing[0]
                if not existing_path.exists() or existing_path.stat().st_size != existing[1]:
                    raise RuntimeError(f'Previously indexed sample missing or changed: {sid}')
                continue
            ys, xs = slice(row, row + 24), slice(col, col + 24)
            native = valid[:, ys, xs]
            target_labels = labels[ys, xs].copy()
            target_labels[~native.all(axis=0)] = VOID
            known = int((target_labels != VOID).sum())
            if known < config['min_label_pixels']:
                continue
            selected_ids = parcel_ids[ys, xs]
            meta = {'id': sid, 'region': g['region'], 'crs': g['crs'],
                    'origin': [west, north], 'pixel_size_m': 10, 'shape': [24, 24],
                    'season': config['season'], 'label_year': config['label_year'],
                    'date_sampling': config.get('date_sampling', 'low_cloud_stratified'),
                    'sampling_seed': config.get('seed', 42),
                    'dates': dates, 'observations': len(dates), 'band_order': BANDS,
                    'reflectance_scale': 10000, 'known_label_pixels': known,
                    'clear_fraction': float(np.isin(scl[:, ys, xs], (4, 5, 6)).mean()),
                    'block': g['id'], 'imagery_cache': observation_key(items, g),
                    'parcel_ids': {str(int(i)): lookup[str(int(i))] for i in np.unique(selected_ids) if i},
                    'source_items': [{'collection': items[i]['collection'], 'id': items[i]['id']} for i in order]}
            sample = {'img': image[:, :, ys, xs].copy(),
                      'doy': np.array([noleap_doy(d) for d in dates], dtype=np.int16),
                      # The semantic loader consumes labels[0]. The other two
                      # annotation planes seen in PASTIS24 are not fabricated.
                      'labels': target_labels[None], 'parcel_ids': selected_ids.copy(),
                      'valid_mask': native.copy(), 'scl': scl[:, ys, xs].copy(), 'metadata': meta}
            relative = Path('pickle24x24') / (sid + '.pickle')
            path = output / relative
            path.parent.mkdir(parents=True, exist_ok=True)
            tmp = path.with_suffix('.tmp')
            with tmp.open('wb') as stream:
                pickle.dump(sample, stream, protocol=4)
            tmp.replace(path)
            connection.execute('INSERT INTO samples VALUES (?,?,?,?)',
                               (sid, relative.as_posix(), path.stat().st_size, json.dumps(meta)))
            count += 1
            emitted += 1
    return emitted, True


def export_summary(connection, output, config, requested, elapsed):
    count = sample_count(connection)
    regions, observations = Counter(), Counter()
    minimum, maximum = observation_limits(config['observations'])
    with (output / 'sample_paths.csv').open('w', newline='') as paths, \
            (output / 'manifest.csv').open('w', newline='') as handle:
        path_writer = csv.writer(paths)
        writer = csv.DictWriter(handle, fieldnames=['id', 'path', 'region', 'crs', 'west', 'north',
                                                   'observations', 'known_label_pixels', 'clear_fraction', 'dates'])
        writer.writeheader()
        for sid, path, _, metadata in connection.execute('SELECT * FROM samples ORDER BY id'):
            m = json.loads(metadata)
            path_writer.writerow([path])
            regions[m['region']] += 1
            observations[str(m['observations'])] += 1
            writer.writerow({'id': sid, 'path': path, 'region': m['region'], 'crs': m['crs'],
                             'west': m['origin'][0], 'north': m['origin'][1], 'observations': m['observations'],
                             'known_label_pixels': m['known_label_pixels'], 'clear_fraction': m['clear_fraction'],
                             'dates': ';'.join(m['dates'])})
    report = {'collected_samples': count, 'requested_target': requested, 'target_met': count >= requested,
              'submission_target': config['target_samples'],
              'submission_target_met': count >= config['target_samples'],
              'sample_definition': 'Distinct geographic 24x24 window with real temporal observations and RPG labels',
              'sample_image_shape': [minimum if minimum == maximum else None, 10, 24, 24],
              'observations_per_sample': dict(observations),
              'configured_observation_range': [minimum, maximum],
              'date_sampling': config.get('date_sampling', 'low_cloud_stratified'),
              'sampling_seed': config.get('seed', 42),
              'sample_label_shape': [1, 24, 24], 'model_input_shape': [60, 24, 24, 11],
              'regions': dict(regions), 'run_elapsed_seconds': round(elapsed, 3),
              'block_status_counts': dict(connection.execute('SELECT status,COUNT(*) FROM blocks GROUP BY status')),
              'sample_bytes': connection.execute('SELECT COALESCE(SUM(bytes),0) FROM samples').fetchone()[0],
              'cloud_filter': 'Clouds recorded in SCL, no strict cloud rejection',
              'minimum_labeled_pixels': config['min_label_pixels'],
              'labels_outside_RPG_or_unmapped': VOID, 'sources': SOURCES,
              'updated_at': datetime.now(timezone.utc).isoformat()}
    write_json(output / 'report.json', report)
    card = f'''# Self-collected Sentinel-2 + RPG crop windows

Actual samples: **{count:,}**. Full submission target: **{config['target_samples']:,}**.
Full target met: **{report['submission_target_met']}**.

Each sample is one distinct 240 m by 240 m geographic window, with
{minimum}..{maximum} actual Sentinel-2 L2A dates from season {config['season']}
and IGN RPG {config['label_year']} parcel labels. Windows are not duplicated
or augmented to increase the count. This is a new collection from public source
imagery and polygons, not a repackaging of PASTIS sample files.

- Raw image: float32 [T,10,24,24], reflectance times 10000.
- Date selection: {config.get('date_sampling', 'low_cloud_stratified')}; seed {config.get('seed', 42)}.
- Band order: {', '.join(BANDS)}.
- Semantic labels: uint8 [1,24,24]; labels[0] is the semantic target.
- Unsupported crop codes, outside parcels and missing native data: void 19.
- Dates, original scene/parcel IDs, CRS and grid origin: sample metadata.
- Cloud SCL is retained; no strict cloud filtering or class balancing.
- Exact code-to-class mapping: class_map.json. Mapping is partial; no claim
  that this reproduces all PASTIS annotation rules or panoptic targets.

sample_paths.csv lists the pickle files relative to this directory.
manifest.csv and report.json contain actual counts and collection metadata.
Every sample is prepared and checked by inspect_dataset using Normalize,
DOY/365.0001, concatenation, cut/pad to 60 and TCHW to THWC.
model_input_example.npy and prepared_sample_example.npz show the final
[60,24,24,11] input and its paired target. Raw samples keep actual observations.
Optional prepared24x24 files contain the final paired inputs and targets for
all samples exported with --prepare-all; they add disk usage.
The dataset has not been evaluated for training quality or prediction accuracy.

Sources and attribution:

- Contains Copernicus Sentinel-2 observations. COG access:
  {SOURCES['imagery']}
- Earth Search catalog: {STAC}
- IGN Registre Parcellaire Graphique, layer RPG.{config['label_year']}:parcelles_graphiques:
  {WFS}
- RPG documentation: {SOURCES['rpg_documentation']}
- Reference tensor format: {SOURCES['reference_format']}

Original catalog records, parcel polygons and per-block scene provenance are
included under cache/. Intermediate image blocks are excluded from the ZIP.
'''
    (output / 'DATASET_CARD.md').write_text(card, encoding='utf-8')
    return report


def discard_block_imagery(items, g, cache):
    """Drop reproducible image intermediates after raw samples are committed."""
    paths = [cache / 'imagery' / (observation_key(items, g) + '.npz')]
    paths += [cache / 'imagery_dates' / (observation_key([item], g) + '.npz') for item in items]
    freed = 0
    for path in paths:
        if path.exists():
            freed += path.stat().st_size
            path.unlink()
    return freed


def prepare_block(client, g, items, config, cache, persist_block):
    """Network half of a block (labels, dates, imagery); runs in a worker thread.

    SQLite and pickle writes stay in the main thread, so workers never share
    the database connection.
    """
    parcels = fetch_parcels(client, g, config['label_year'], cache)
    labels, ids, lookup, codes = rasterize_parcels(parcels, g)
    if not np.any(labels != VOID):
        return {'status': 'no_labels', 'message': 'No supported RPG crop labels'}
    selected = select_observations(items, g, config['observations'],
                                   sampling=config.get('date_sampling', 'low_cloud_stratified'),
                                   seed=config.get('seed', 42))
    if not selected:
        return {'status': 'no_coverage', 'message': 'No single tile with enough distinct dates covers block'}
    image, valid, scl = read_observations(selected, g, cache, config['workers'], persist_block)
    return {'status': 'ready', 'selected': selected, 'image': image, 'valid': valid, 'scl': scl,
            'labels': labels, 'ids': ids, 'lookup': lookup, 'codes': codes}


def collect(config, output, target_samples=None, max_blocks=None, keep_completed_imagery=True,
            block_workers=3, time_budget_s=None, summary_every=10):
    """Download up to `block_workers` blocks concurrently and export them as they finish.

    `time_budget_s` stops submitting new blocks after that many seconds; blocks
    already in flight are still exported. Rerun with the same output to resume.
    """
    validate_config(config)
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    requested = target_samples if target_samples is not None else config['target_samples']
    if not isinstance(requested, int) or requested < 1 or (max_blocks is not None and max_blocks < 1):
        raise ValueError('Target and max_blocks must be positive.')
    if not isinstance(block_workers, int) or not 1 <= block_workers <= 8:
        raise ValueError('block_workers must be 1..8.')
    saved = output / 'effective_config.json'
    effective = {'config': config, 'class_map': CROP_MAP, 'format_version': 1}
    if saved.exists() and json.loads(saved.read_text()) != effective:
        raise ValueError('Config differs from this output. Use a new output directory to avoid mixing datasets.')
    write_json(saved, effective)
    write_json(output / 'class_map.json', {'codes': CROP_MAP, 'classes': CLASS_NAMES,
               'void': VOID, 'note': 'Conservative mapping; not a reproduction of the original PASTIS labels.'})
    cache = output / 'cache'
    connection = open_index(output)
    # Detect deleted/truncated samples before trusting a resumed count.
    for sid, relative, size in connection.execute('SELECT id,path,bytes FROM samples'):
        path = output / relative
        if not path.exists() or path.stat().st_size != size:
            connection.close()
            raise RuntimeError(f'Indexed sample missing or changed: {sid}')
    finished = {row[0] for row in connection.execute(
        "SELECT id FROM blocks WHERE status IN ('complete','no_labels','no_coverage')")}
    started, attempted, consecutive_errors, handled = time.monotonic(), 0, 0, 0
    stop_reason = None
    try:
        with httpx.Client(timeout=httpx.Timeout(60, connect=15), follow_redirects=True,
                          limits=httpx.Limits(max_connections=32)) as client, \
                ThreadPoolExecutor(max_workers=block_workers) as pool:
            catalogs = {r['id']: search_region(client, r, config, cache) for r in config['regions']}
            candidates = (g for g in interleaved_blocks(config) if g['id'] not in finished)
            in_flight = {}
            while True:
                count = sample_count(connection)
                if stop_reason is None:
                    if count >= requested:
                        stop_reason = 'target_reached'
                    elif max_blocks is not None and attempted >= max_blocks:
                        stop_reason = 'max_blocks'
                    elif time_budget_s is not None and time.monotonic() - started >= time_budget_s:
                        stop_reason = 'time_budget'
                    elif consecutive_errors >= 3:
                        stop_reason = 'three_consecutive_errors'
                while stop_reason is None and len(in_flight) < block_workers:
                    g = next(candidates, None)
                    if g is None:
                        stop_reason = 'no_more_blocks'
                        break
                    attempted += 1
                    in_flight[pool.submit(prepare_block, client, g, catalogs[g['region']], config,
                                          cache, keep_completed_imagery)] = g
                    if max_blocks is not None and attempted >= max_blocks:
                        break
                if not in_flight:
                    break
                done, _ = wait(in_flight, return_when=FIRST_COMPLETED)
                for future in done:
                    g = in_flight.pop(future)
                    result = None
                    try:
                        result = future.result()
                        status, message = result['status'], result.get('message', '')
                        if status == 'ready':
                            selected = result['selected']
                            n, complete = export_block(connection, output, g, selected, result['image'],
                                                       result['valid'], result['scl'], result['labels'],
                                                       result['ids'], result['lookup'], config, requested)
                            status, message = ('complete' if complete else 'partial'), f'{n} samples exported'
                            write_json(cache / 'provenance' / (g['id'] + '.json'),
                                       {'grid': g, 'items': selected, 'rpg_codes': result['codes'],
                                        'source': SOURCES})
                        connection.execute('INSERT OR REPLACE INTO blocks VALUES (?,?,?)', (g['id'], status, message))
                        connection.commit()
                        if status == 'complete' and not keep_completed_imagery:
                            discard_block_imagery(result['selected'], g, cache)
                        consecutive_errors = 0
                    except Exception as error:
                        # Samples already written in this block remain indexed;
                        # a retry sees their geographic IDs and does not duplicate.
                        status, message = 'error', f'{type(error).__name__}: {error}'
                        connection.execute('INSERT OR REPLACE INTO blocks VALUES (?,?,?)', (g['id'], status, message))
                        connection.commit()
                        consecutive_errors += 1
                    result = None  # free block arrays before waiting on the next one
                    handled += 1
                    elapsed = time.monotonic() - started
                    collected = sample_count(connection)
                    emit('block_done', id=g['id'], status=status, message=message[:200],
                         collected=collected, target=requested, elapsed_min=round(elapsed / 60, 1))
                    if handled % summary_every == 0:
                        export_summary(connection, output, config, requested, elapsed)
            if stop_reason == 'three_consecutive_errors':
                emit('paused', reason='Three consecutive block errors; fix the source or network and resume.')
        report = export_summary(connection, output, config, requested, time.monotonic() - started)
    finally:
        connection.commit()
        export_summary(connection, output, config, requested, time.monotonic() - started)
        connection.close()
    emit('finished', collected=report['collected_samples'], requested=requested, stop_reason=stop_reason,
         submission_target_met=report['submission_target_met'])
    return report


def inspect_dataset(output, preview=True, save_prepared=False):
    output = Path(output)
    with (output / 'sample_paths.csv').open(newline='') as stream:
        paths = [row[0] for row in csv.reader(stream) if row]
    count, shapes, classes, seen = 0, Counter(), Counter(), set()
    prepared_files, observation_counts = [], Counter()
    first = None
    for relative in paths:
        # Only load files generated by this collector or another trusted source.
        with (output / relative).open('rb') as stream:
            sample = pickle.load(stream)
        m = sample['metadata']
        key = (m['crs'], *m['origin'], m['label_year'])
        if key in seen:
            raise ValueError('Duplicate geographic window: ' + relative)
        seen.add(key)
        image, labels, doy = sample['img'], sample['labels'], sample['doy']
        if image.shape != (len(doy), 10, 24, 24) or labels.shape != (1, 24, 24):
            raise ValueError('Invalid sample shape: ' + relative)
        if not np.isfinite(image).all() or len(set(m['dates'])) != len(doy):
            raise ValueError('Nonfinite image or duplicate date: ' + relative)
        if not np.isin(labels, np.arange(20)).all():
            raise ValueError('Invalid label ID: ' + relative)
        if np.sum(labels != VOID) < 1:
            raise ValueError('No actual mapped RPG label: ' + relative)
        prepared = prepare_sample(sample)
        inputs = prepared['inputs']
        n = prepared['seq_lengths']
        if (inputs.shape != (60, 24, 24, 11) or inputs.dtype != np.float32
                or not np.isfinite(inputs).all() or not (inputs[n:] == 0).all()):
            raise ValueError('Invalid prepared input or temporal padding: ' + relative)
        if save_prepared:
            prepared_path = Path('prepared24x24') / (Path(relative).stem + '.npz')
            save_prepared_sample(output / prepared_path, prepared)
            prepared_files.append(prepared_path.as_posix())
        observation_counts[str(len(doy))] += 1
        shapes[str(list(image.shape))] += 1
        values, counts = np.unique(labels, return_counts=True)
        classes.update({str(int(v)): int(n) for v, n in zip(values, counts)})
        count += 1
        if first is None:
            first = sample
        if count % 1000 == 0:
            emit('preprocessing', prepared=count, total=len(paths))
    result = {'verified_samples': count, 'unique_geographic_windows': len(seen),
              'image_shapes': dict(shapes), 'label_pixel_counts': dict(classes),
              'fully_preprocessed_samples': count, 'observations_per_sample': dict(observation_counts),
              'prepared_pairs_exported': len(prepared_files)}
    if save_prepared:
        with (output / 'prepared_paths.csv').open('w', newline='') as stream:
            csv.writer(stream).writerows([path] for path in prepared_files)
    if first is not None and preview:
        paired = prepare_sample(first)
        tensor = paired['inputs']
        np.save(output / 'model_input_example.npy', tensor, allow_pickle=False)
        save_prepared_sample(output / 'prepared_sample_example.npz', paired)
        write_json(output / 'pipeline_example.json', paired['pipeline'])
        draw_preview(first, output / 'preview.png')
        result.update(model_input_shape=list(tensor.shape),
                      zero_padding_correct=bool((tensor[len(first['doy']):] == 0).all()))
    write_json(output / 'verification.json', result)
    return result


def draw_preview(sample, path):
    t = len(sample['doy'])
    shown = np.linspace(0, t - 1, min(t, 5), dtype=int).tolist()
    canvas = Image.new('RGB', ((len(shown) + 1) * 192, 228), 'white')
    draw = ImageDraw.Draw(canvas)
    for col, i in enumerate(shown):
        rgb = np.uint8(np.clip(sample['img'][i, [2, 1, 0]] / 3000, 0, 1) ** (1 / 1.3) * 255)
        image = Image.fromarray(rgb.transpose(1, 2, 0)).resize((192, 192), Image.Resampling.NEAREST)
        canvas.paste(image, (col * 192, 24))
        draw.text((col * 192 + 4, 4), sample['metadata']['dates'][i], fill='black')
    palette = np.array([[int(hashlib.sha256(str(i).encode()).hexdigest()[j:j+2], 16)
                         for j in (0, 2, 4)] for i in range(20)], dtype=np.uint8)
    palette[VOID] = [180, 180, 180]
    label_image = Image.fromarray(palette[sample['labels'][0]]).resize((192, 192), Image.Resampling.NEAREST)
    canvas.paste(label_image, (len(shown) * 192, 24))
    draw.text((len(shown) * 192 + 4, 4), 'RPG labels; gray = void', fill='black')
    canvas.save(path)


def pack_dataset(output, destination=None):
    output = Path(output)
    destination = Path(destination) if destination else output.parent / (output.name + '.zip')
    with (output / 'sample_paths.csv').open(newline='') as stream:
        sample_files = [output / row[0] for row in csv.reader(stream) if row]
    names = ['sample_paths.csv', 'manifest.csv', 'report.json', 'verification.json', 'index.sqlite3', 'DATASET_CARD.md',
             'effective_config.json', 'class_map.json', 'preview.png', 'model_input_example.npy',
             'live_validation.json', 'notebook_validation.json', 'prepared_sample_example.npz',
             'pipeline_example.json', 'prepared_paths.csv']
    files = sample_files + [output / name for name in names if (output / name).exists()]
    files += sorted((output / 'prepared24x24').glob('*.npz'))
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + '.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        for path in files:
            archive.write(path, path.relative_to(output).as_posix())
        # Each sample contains IDs/dates; retain the exact source metadata and
        # actual parcel polygons so the paired sample provenance can be audited.
        for folder in ['catalog', 'parcels', 'provenance']:
            for path in sorted((output / 'cache' / folder).glob('*')):
                archive.write(path, path.relative_to(output).as_posix())
    temporary.replace(destination)
    return destination


def save_notebook_outputs(output, export_dir, max_output_bytes=None):
    """Save only a resumable ZIP and readable reports in the cloud output dir.

    Collection, raw windows and intermediate images remain in scratch space.
    This does not upload anything or publish to Kaggle/Drive.
    """
    output, export_dir = Path(output), Path(export_dir)
    if export_dir.resolve().is_relative_to(output.resolve()):
        raise ValueError('Export directory must be outside the collection directory.')
    export_dir.mkdir(parents=True, exist_ok=True)
    # Write the ZIP once, directly in the export dir; avoids a multi-GB copy
    # from scratch to /kaggle/working.
    archive = pack_dataset(output, export_dir / (output.name + '.zip'))
    report_files = [output / name for name in ['report.json', 'verification.json',
                    'DATASET_CARD.md', 'manifest.csv', 'class_map.json',
                    'pipeline_example.json', 'preview.png'] if (output / name).exists()]
    total = archive.stat().st_size + sum(p.stat().st_size for p in report_files)
    if max_output_bytes is not None and total > max_output_bytes:
        shutil.move(str(archive), str(output.parent / archive.name))
        raise RuntimeError(f'Export needs {total:,} bytes, over the {max_output_bytes:,}-byte limit. '
                           'The ZIP is safe in scratch space; export a smaller collection stage.')
    for path in report_files:
        shutil.copy2(path, export_dir / path.name)
    return archive


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--config', type=Path, default=Path(__file__).with_name('config.json'))
    parser.add_argument('--output', type=Path, default=Path('data_collection/outputs/pastis24_collected'))
    parser.add_argument('--target', type=int, help='Stage limit; report still records the submission target')
    parser.add_argument('--max-blocks', type=int, help='Bound this run; repeat command to continue')
    parser.add_argument('--plan', action='store_true', help='Geometry/count plan only; no network or downloads')
    parser.add_argument('--verify', action='store_true', help='Verify existing output without downloading')
    parser.add_argument('--prepare-all', action='store_true', help='Also export all final input-target pairs as compressed NPZ')
    parser.add_argument('--discard-completed-cache', action='store_true', help='Remove image intermediates after completing each block')
    parser.add_argument('--pack', action='store_true', help='Also create a ZIP without intermediate imagery cache')
    args = parser.parse_args()
    config = json.loads(args.config.read_text())
    if args.plan:
        print(json.dumps(collection_plan(config), indent=2))
        return
    if args.verify:
        print(json.dumps(inspect_dataset(args.output, save_prepared=args.prepare_all), indent=2))
    else:
        collect(config, args.output, args.target, args.max_blocks,
                keep_completed_imagery=not args.discard_completed_cache)
        print(json.dumps(inspect_dataset(args.output, save_prepared=args.prepare_all), indent=2))
    if args.pack:
        print('Dataset ZIP:', pack_dataset(args.output))


## 3. Cấu hình và quy mô
Mùa `2020-2021`, nhãn `RPG.2021`, `observations=[8,12]`, `date_sampling="random"`, `seed=42`, mục tiêu `25000` mẫu. Đổi seed/mùa/vùng thì dùng output mới; không thay config của bộ đang thu dở.

Tham số luồng:
- `BLOCK_WORKERS`: số khối tải song song (mỗi khối 16×16 cửa sổ). `CONFIG["workers"]`: số băng COG đọc song song trong một khối. Tổng kết nối ≈ tích hai số. Lỗi mạng/429 nhiều thì giảm.
- `TIME_BUDGET_HOURS`: sau mốc này không nhận khối mới, các khối đang tải vẫn được ghi. Phiên Kaggle tối đa 12 giờ; để dư ~1 giờ cho kiểm tra và đóng gói.
- `SAVE_ALL_PREPARED=False`: vẫn chạy tiền xử lý cho mọi mẫu khi kiểm tra, chỉ lưu một cặp minh họa.

25.000 mẫu raw (8–12 ngày, float32) khoảng 5–7 GB. Trên Kaggle, mẫu/cache ghi ở `/kaggle/temp/crop_collection`, ZIP và báo cáo ghi thẳng vào `/kaggle/working/submission` (giới hạn 20 GB). Cache ảnh của khối hoàn tất bị xóa ngay.

Chưa đủ 25.000 khi hết giờ: Save Version, tạo Kaggle Dataset từ ZIP, phiên sau điền `RESTORE_ZIP` rồi chạy lại cùng cấu hình.


In [ ]:
CONFIG = {
    "season": "2020-2021",
    "label_year": 2021,
    "target_samples": 25000,
    "observations": [8, 12],
    "date_sampling": "random",
    "seed": 42,
    "pixel_size_m": 10,
    "window_size": 24,
    "block_windows": 16,
    "workers": 8,
    "min_label_pixels": 1,
    "regions": [
        {"id": "normandy", "center": [-0.433, 49.25], "size_m": 40000},
        {"id": "beauce", "center": [1.5, 48.1], "size_m": 40000},
        {"id": "toulouse", "center": [1.15, 43.75], "size_m": 40000},
        {"id": "alsace", "center": [7.45, 48.2], "size_m": 40000},
    ],
}
TARGET_THIS_RUN = CONFIG["target_samples"]
MAX_BLOCKS = None          # đặt số nhỏ (vd 2) để chạy thử
BLOCK_WORKERS = 3          # số khối tải song song
TIME_BUDGET_HOURS = 10.5   # Kaggle giới hạn phiên 12 giờ
SAVE_ALL_PREPARED = False  # True: xuất NPZ input–target cho mọi mẫu, tốn thêm ổ đĩa
KEEP_COMPLETED_IMAGERY = False  # Dọn cache sau khi đã ghi xong mẫu của cả khối
IS_KAGGLE = Path("/kaggle/working").is_dir()
MAX_EXPORT_BYTES = 20_000_000_000 if IS_KAGGLE else None
if IS_KAGGLE:
    base = Path("/kaggle/temp/crop_collection")
    EXPORT_DIR = Path("/kaggle/working/submission")
elif Path("/content").is_dir():
    base = Path("/content/crop_collection")
    EXPORT_DIR = Path("/content/submission")
else:
    base = Path.cwd()/"outputs"
    EXPORT_DIR = base/"submission"
OUTPUT = base/"collected_pastis24_2020_2021"
COLAB_DRIVE_EXPORT = None  # Ví dụ: "/content/drive/MyDrive/crop_collection"
if COLAB_DRIVE_EXPORT:
    from google.colab import drive
    drive.mount("/content/drive")
    EXPORT_DIR = Path(COLAB_DRIVE_EXPORT)
print(json.dumps(collection_plan(CONFIG),indent=2))
print("This run target:", TARGET_THIS_RUN, "Block workers:", BLOCK_WORKERS,
      "Time budget (h):", TIME_BUDGET_HOURS)
print("Scratch:", OUTPUT, "Saved output:", EXPORT_DIR)
existing = base
while not existing.exists():
    existing = existing.parent
print("Scratch disk free (GB):", round(shutil.disk_usage(existing).free/1e9, 2))


## 4. Phục hồi lượt trước (tùy chọn)
Attach ZIP của bộ **cùng config mùa 2020-2021, 8–12 ngày** bằng Kaggle Dataset rồi điền `RESTORE_ZIP`. ZIP giữ chỉ mục/catalog/provenance để chạy tiếp. Không trộn ZIP của mùa khác (vd 2018-2019) vào bộ này. Thay seed, vùng, khoảng ngày hoặc bảng mã cần một output mới.


In [ ]:
RESTORE_ZIP = None  # ví dụ: "/kaggle/input/my-collection/collected_pastis24_2020_2021.zip"
if RESTORE_ZIP:
    if (OUTPUT/"index.sqlite3").exists():
        raise RuntimeError("Output already has an index; do not overwrite it with an older ZIP")
    OUTPUT.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(RESTORE_ZIP) as archive:
        for info in archive.infolist():
            destination=(OUTPUT/info.filename).resolve()
            if not destination.is_relative_to(OUTPUT.resolve()):
                raise ValueError("Unsafe path in archive")
        archive.extractall(OUTPUT)
    print("Restored:",OUTPUT)

## 5. Thu thập ảnh và nhãn thật
Lấy catalog và polygon RPG trực tiếp qua mạng; chọn ngẫu nhiên trong các ngày có ảnh phủ trọn khối, bỏ bản trùng ngày và ưu tiên bản reprocess mới. Đọc phần COG cần thiết, áp dụng scale/offset, bilinear cho băng phổ và nearest cho SCL. Lưu cache từng ngày hoàn tất để phục hồi sau gián đoạn.

Nhãn đúng năm/vùng được rasterize và cắt cùng cửa sổ ảnh. Giữ kiểm tra native data; không lọc mây khắt khe. ID địa lý và SQLite ngăn đếm trùng mẫu.


In [ ]:
report = collect(CONFIG, OUTPUT, target_samples=TARGET_THIS_RUN, max_blocks=MAX_BLOCKS,
                 keep_completed_imagery=KEEP_COMPLETED_IMAGERY, block_workers=BLOCK_WORKERS,
                 time_budget_s=TIME_BUDGET_HOURS * 3600 if TIME_BUDGET_HOURS else None)
print(json.dumps({k: report[k] for k in ["collected_samples", "requested_target", "target_met",
                  "regions", "observations_per_sample", "block_status_counts",
                  "sample_bytes", "run_elapsed_seconds"]}, indent=2))

## 6. Thực hiện pipeline slide 2 cho mọi mẫu
`inspect_dataset()` đọc từng mẫu và gọi `prepare_sample()`: Normalize 10 kênh → DOY/365.0001 thành `[T,1,24,24]` → ghép `[T,11,24,24]` → cut/pad `[60,11,24,24]` → THWC `[60,24,24,11]`. Target semantic và mask void được giữ cùng mẫu.

Kiểm tra input hữu hạn, padding đúng zero, số ngày/vị trí không trùng, nhãn hợp lệ. Xuất `pipeline_example.json` ghi kích thước từng bước. Bộ thu chưa được đánh giá chất lượng để train.


In [ ]:
verification = inspect_dataset(OUTPUT, save_prepared=SAVE_ALL_PREPARED)
print(json.dumps(verification,indent=2))
if (OUTPUT/"preview.png").exists():
    try:
        from IPython.display import display
    except ImportError:
        print("Preview:", OUTPUT/"preview.png")
    else:
        display(Image.open(OUTPUT/"preview.png"))
with (OUTPUT/"sample_paths.csv").open() as stream:
    first = next(csv.reader(stream),None)
if first:
    with (OUTPUT/first[0]).open("rb") as stream:
        example = pickle.load(stream)
    print("img:",example["img"].shape,example["img"].dtype)
    print("labels:",example["labels"].shape,"DOY:",example["doy"])
    print("dates:",example["metadata"]["dates"])
    paired = prepare_sample(example)
    print("Paired input:", paired["inputs"].shape, "target:", paired["labels"].shape)
    print(json.dumps(paired["pipeline"], indent=2))

## 7. Đóng gói và trạng thái bài nộp
ZIP gồm raw pickle, báo cáo, pipeline/cặp minh họa, polygon/provenance và chỉ mục để chạy tiếp. Nếu bật `SAVE_ALL_PREPARED`, có thêm `prepared24x24/` và `prepared_paths.csv` chứa các cặp đã xử lý.

Bài đủ số lượng khi `submission_target_met=true` (25.000 mẫu mùa 2020-2021). Save Version có output sau khi chạy, rồi tạo Kaggle Dataset từ ZIP nếu muốn phục hồi ở phiên khác.

**Cách nộp task 2:** tạo Kaggle Dataset public chứa ZIP dữ liệu và report, liên kết notebook public giải thích nguồn, cách chọn ngày, hai nhánh ảnh–nhãn và kiểm tra pipeline. Chỉ báo đủ số mẫu khi report xác nhận. Nộp link này trong PDF tổng hợp theo yêu cầu thầy.

Tài liệu nền tảng: [Kaggle Notebooks — output tối đa 20 GB](https://www.kaggle.com/docs/notebooks), [Colab FAQ — giới hạn phiên và lưu Drive](https://research.google.com/colaboratory/faq.html).


In [ ]:
archive = save_notebook_outputs(OUTPUT, EXPORT_DIR, max_output_bytes=MAX_EXPORT_BYTES)
print("ZIP:",archive,"MiB:",round(archive.stat().st_size/1024**2,2))
final_report=json.loads((OUTPUT/"report.json").read_text())
print("Collected:",final_report["collected_samples"],"/",final_report["submission_target"])
print("Submission target met:",final_report["submission_target_met"])
if not final_report["submission_target_met"]:
    print("INCOMPLETE: continue collection; do not report it as the full target.")